# Train Deep Learning (2xT4)

W&B required. Primary CNN; ViT backup. Run bootstrap first.

## 0) Bootstrap (Kaggle / Colab)

Run once per session: clone or pull this repo, `cd` into it, install deps. Locally you can skip clone if you already opened the notebook inside the repo.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Classification.git"
REPO_DIR = "VN-Traffic-Sign-Classification"

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str) -> None:
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

if (repo_path / ".git").exists():
    sh(f'git -C "{repo_path}" pull --ff-only')
elif (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using existing repo at", repo_path)
    try:
        sh("git pull --ff-only")
    except subprocess.CalledProcessError:
        print("git pull skipped/failed — continue with local tree")
else:
    sh(f'git clone {REPO_URL} "{repo_path}"')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

req = repo_path / "requirements-kaggle.txt"
if req.exists():
    sh(f'pip install -q -r "{req}"')
else:
    sh("pip install -q -r " + str(repo_path / "requirements.txt"))
sh("pip install -q -e .")
print("bootstrap OK")


## 1) Overrides + train

In [ ]:
OVERRIDES = {
  # "model": {"backbone": "efficientnet_b0", "family": "cnn"},
  # "model": {
  #   "backbone": "vit_tiny_patch16_224",
  #   "family": "transformer",
  #   "transformer": {"use_flash_attn": False},
  # },
  # "train": {"batch_size": 64, "epochs": 20},
}
from vn_tsc.config.resolve import resolve_config
cfg = resolve_config("configs/pipelines/dl.yaml", "configs/shared.yaml", "configs/runtime/kaggle_2xt4.yaml", OVERRIDES)
print(cfg.get("pipeline"), cfg.get("model"))
# TODO(team-dl)